In [1]:
import os
from azure.identity import DefaultAzureCredential
from dotenv import load_dotenv
from azure.ai.projects import AIProjectClient
from azure.ai.textanalytics import TextAnalyticsClient
from azure.core.credentials import AzureKeyCredential

load_dotenv()

foundry_project_endpoint= os.getenv("FOUNDRY_PROJECT_ENDPOINT")
model_deployment_name= os.getenv("MODEL_DEPLOYMENT_NAME")
azure_lang_endpoint=os.getenv("AZURE_LANGUAGE_ENDPOINT")
azure_lang_endpoint_api_key=os.getenv("AZURE_LANGUAGE_ENDPOINT_API_KEY")

Setting Up the Foundry Project Client


In [3]:
project_client=AIProjectClient(
    endpoint=foundry_project_endpoint,
    credential=DefaultAzureCredential()
)

Create a Text Analytics Client

In [7]:
ta_credential=AzureKeyCredential(azure_lang_endpoint_api_key)

text_analytics_client = TextAnalyticsClient(
    endpoint=azure_lang_endpoint,
    credential=ta_credential
)

In [8]:
openai_client = project_client.get_openai_client()

In [10]:
def PII_Text_Redaction(input_text:str)->str:
    documents=[
        input_text
    ]

    response= text_analytics_client.recognize_pii_entities(documents, language="en")
    
    result = [doc for doc in response if not doc.is_error]
    for doc in result:
        redacted_text = doc.redacted_text
        for entity in doc.entities:
            print("Entity {}".format(entity.text))
            print("Category {}".format(entity.category))
            print("Confidence Score {}".format(entity.confidence_score))
            print("offset {}".format(entity.offset))
            print("Length {}".format(entity.length))
        return redacted_text    
            
            
            
            

In [13]:
from azure.ai.projects.models import PromptAgentDefinition

agent_name="travel-assistant-agent"

agent = project_client.agents.create_version(
    agent_name=agent_name,
    definition=PromptAgentDefinition(
        model=model_deployment_name,
        instructions="You are a travel assistant, help users to plan their trips, find hotels, flights, and provide best option among all"
    ),
)

print(f'Agent create (id: {agent.id}, name: {agent.name}, version: {agent.version})')

Agent create (id: travel-assistant-agent:1, name: travel-assistant-agent, version: 1)


In [14]:
conversation = openai_client.conversations.create()

In [18]:
chat = True

while chat:
    user_input= input("You: ")
    if user_input.lower() in ["exit", "quit"]:
        chat= False
        print("Exiting chat. Goodbye!!")
    else:
        redacted_text= PII_Text_Redaction(user_input)
        print(f"Redacted input: {redacted_text}")

        response = openai_client.responses.create(
            conversation=conversation.id,
            extra_body={
                "agent":{
                    "name":agent_name,
                    "type":"agent_reference"
                }
            },
            input=redacted_text
        )
        
        print(f'Agent: {response.output_text}')


Entity deepanshu
Category Person
Confidence Score 1.0
offset 11
Length 9
Entity deepans@gmail.com
Category Email
Confidence Score 0.8
offset 45
Length 17
Redacted input: my name is ********* and my email address is ***************** can you help me to find the best flight from delhi to mum?
Agent: Hello! I can definitely help you find the best flight from Delhi to Mumbai. Could you please provide a few more details?

1. Your preferred travel dates (departure and return, if round-trip)
2. Any preferred airlines or budget considerations
3. Do you want economy, business, or first class?
4. Any specific time preferences for the flight (morning, afternoon, evening)?

Once I have this information, I can find the best options for you!
Entity 29 june 2026
Category DateTime
Confidence Score 1.0
offset 27
Length 12
Entity 1 july
Category DateTime
Confidence Score 1.0
offset 65
Length 6
Redacted input: my preferred date would be ************ and return date would be ******
Agent: Thanks for provi